In [15]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import requests
import numpy as np
import psycopg2

In [16]:
DB_CONFIG = {
    "host" : "localhost",
    "port" : "5432",
    "dbname" : "rag_database",
    "user" : "rag_user",
    "password" : "rag@password-3112"
}

In [17]:
def get_embedding(text):
    r = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model":"nomic-embed-text", "prompt":text}
    )
    return np.array(r.json()["embedding"])

In [18]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [19]:
def insert_chunk(conn, content, source):
    vec = get_embedding(content)
    vec_str = "[" + ",".join(f"{x:.6f}" for x in vec)+"]"

    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO chunks(content, embedding, source) VALUES (%s, %s, %s) RETURNING id", (content, vec_str, source)
        )
        new_id = cur.fetchone()[0]
        conn.commit()
        return new_id

In [20]:
def search(conn, query, top_k=3):
    q_vec = get_embedding(query)
    q_str = "[" + ",".join(f"{x:.6f}" for x in q_vec) + "]"
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, content, source,
                   1 - (embedding <=> %s::vector) AS similarity
            FROM chunks
            ORDER BY embedding <=> %s::vector
            LIMIT %s
        """, (q_str, q_str, top_k))
        return cur.fetchall()

In [21]:
if __name__ == "__main__":
    conn = psycopg2.connect(**DB_CONFIG)

    # Insert 3 documents
    docs = [
        ("The service will not start after the last deployment.", "doc-1"),
        ("Quarterly revenue grew by 12% in the north region.",    "doc-2"),
        ("Container keeps restarting in a crash loop.",            "doc-3"),
    ]
    for content, source in docs:
        new_id = insert_chunk(conn, content, source)
        print(f"Inserted id={new_id}: {content[:50]}...")

    # Search
    print("\n--- Search: 'deployment failure' ---")
    for row in search(conn, "deployment failure"):
        id_, content, source, sim = row
        print(f"  {sim:.3f}  [{source}]  {content[:60]}")

    conn.close()

Inserted id=19: The service will not start after the last deployme...
Inserted id=20: Quarterly revenue grew by 12% in the north region....
Inserted id=21: Container keeps restarting in a crash loop....

--- Search: 'deployment failure' ---
  0.683  [doc-1]  The service will not start after the last deployment.
  0.683  [doc-1]  The service will not start after the last deployment.
  0.683  [doc-1]  The service will not start after the last deployment.
